# 03_pc_<agreement>_<source>_to_<target>

Pipeline notebook: run-all-safe deterministic enforcement.

In [ ]:
# In Fabric, run this first:
# %run 00_env_config

In [ ]:
from fabricops_kit import *

In [ ]:
LOCAL_NOTEBOOK_NAME_FALLBACK = "03_pc_agreement_source_to_target"  # local-dev only
CONFIG = load_fabric_config(CONFIG)
validate_notebook_name(local_fallback_name=LOCAL_NOTEBOOK_NAME_FALLBACK)
assert_notebook_name_valid(local_fallback_name=LOCAL_NOTEBOOK_NAME_FALLBACK)
bootstrap_fabric_env(config=CONFIG, environment="dev")
RUN_ID = generate_run_id(prefix="pc")
runtime_context = build_runtime_context(dataset_name="target_dataset", environment="dev", source_table="source.table", target_table="product.table", run_id=RUN_ID)

In [ ]:
ENVIRONMENT = "dev"
SOURCE_LAYER = "source"
TARGET_LAYER = "product"
SOURCE_TABLE = "TODO_source_table"
TARGET_TABLE = "TODO_target_table"
TARGET_KIND = "lakehouse"
WRITE_MODE = "overwrite"

In [ ]:
contract = load_data_contract("contracts/examples/normalized_data_product_contract.yml")  # TODO replace

In [ ]:
source_house = get_path(ENVIRONMENT, SOURCE_LAYER, config=CONFIG)
target_house = get_path(ENVIRONMENT, TARGET_LAYER, config=CONFIG)
df_source = lakehouse_table_read(source_house, SOURCE_TABLE)

In [ ]:
required_columns = set(contract.get("source_schema", {}).keys())
missing_columns = sorted(required_columns - set(df_source.columns))
if missing_columns:
    raise ValueError(f"Fail fast: missing required source columns: {missing_columns}")

In [ ]:
# TODO: approved deterministic transformation logic
df_transformed = df_source

In [ ]:
dq_results = run_quality_rules(df_transformed, contract.get("quality_rules", []), dataset_name="target_dataset")
if dq_results.get("failed"):
    raise ValueError("Fail fast: quality rules failed.")

In [ ]:
df_standard = add_datetime_features(df_transformed)  # optional


In [ ]:
df_standard = add_audit_columns(df_standard, run_id=RUN_ID)
df_standard = add_hash_columns(df_standard)
tech_cols = default_technical_columns()

In [ ]:
expected_output_cols = set(contract.get("target_schema", {}).keys())
actual_output_cols = set(c for c in df_standard.columns if c not in tech_cols)
if expected_output_cols and expected_output_cols != actual_output_cols:
    raise ValueError("Fail fast: output contract mismatch.")

In [ ]:
if TARGET_KIND == "lakehouse":
    lakehouse_table_write(df_standard, target_house, TARGET_TABLE, mode=WRITE_MODE)
else:
    warehouse_write(df_standard, target_house, schema="dbo", table=TARGET_TABLE, mode=WRITE_MODE)

In [ ]:
df_written = lakehouse_table_read(target_house, TARGET_TABLE) if TARGET_KIND == "lakehouse" else warehouse_read(target_house, schema="dbo", table=TARGET_TABLE)
if df_written.count() == 0:
    raise ValueError("Fail fast: target write produced zero rows.")

In [ ]:
output_profile = generate_metadata_profile(df_written, dataset_name="target_dataset")
output_profile_rows = profile_dataframe_to_metadata(df_written, dataset_name="target_dataset")

In [ ]:
quality_records = build_quality_result_records(dq_results, dataset_name="target_dataset", run_id=RUN_ID)
dataset_record = build_dataset_run_record(dataset_name="target_dataset", run_id=RUN_ID, row_count=df_written.count())
write_metadata_records([*quality_records, dataset_record, *output_profile_rows], config=CONFIG)

In [ ]:
lineage = build_lineage_from_notebook_code(__file__, dataset_name="target_dataset", run_id=RUN_ID)
lineage_records = build_lineage_records(dataset_name="target_dataset", run_id=RUN_ID, lineage_steps=lineage.get("steps", []))
print(build_lineage_handover_markdown(dataset_name="target_dataset", run_id=RUN_ID, lineage_result=lineage))